# 13 – Seven-variable Sensitivity Analysis (Appendix B)

Re-runs the three signal-using models on a reduced **seven-variable** feature set
(heart rate, systolic/diastolic/mean blood pressure, respiratory rate, temperature,
oxygen saturation) instead of the sixteen used in the main analysis, to check that the
findings are not specific to the expanded feature set.

Reads the same `hourly_vitals.csv` as the main pipeline but selects only the seven
vital-sign columns (plus their masks), so no re-extraction is needed. The text modality
and cohort labels are identical to the main analysis.

**Run after notebooks 05, 06 and 09b** — uses hourly_vitals.csv,
modelling_cohort_sepsis_mortality.csv, text_hourly_cls_mbert.npz.

**Produces:** printed 7-variable results (Table B.1). No files written.

MIMIC-III data not included (PhysioNet DUA); see README.


In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import random
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# --- Load data: SAME files as main pipeline, but SEVEN vital signs only ---
tz = np.load(data_path("text_hourly_cls_mbert.npz"), allow_pickle=True)
X_text_all, has_note_all, text_ids = tz["X_text"], tz["has_note"], tz["stay_ids"]

vitals = pd.read_csv(data_path("hourly_vitals.csv"))
cohort = pd.read_csv(data_path("modelling_cohort_sepsis_mortality.csv"))
labels = cohort[["ICUSTAY_ID","mortality_after_24h"]].drop_duplicates()

# --- seven-variable set (Appendix B) ---
VITAL_COLS = ["heart_rate","sbp","dbp","map","resp_rate","temperature","spo2"]
MASK_COLS  = [c + "_observed" for c in VITAL_COLS]
FEAT_COLS  = VITAL_COLS + MASK_COLS
print("feature columns:", len(FEAT_COLS), "(7 vitals + 7 masks)")

stay_ids = text_ids
y = labels.set_index("ICUSTAY_ID")["mortality_after_24h"]
vitals = vitals[vitals["ICUSTAY_ID"].isin(set(stay_ids))].copy()
full_idx = pd.MultiIndex.from_product([stay_ids, range(24)], names=["ICUSTAY_ID","ICU_HOUR"])
vit = vitals.set_index(["ICUSTAY_ID","ICU_HOUR"]).reindex(full_idx)[FEAT_COLS]
X_sig_raw = vit[FEAT_COLS].values.reshape(len(stay_ids), 24, len(FEAT_COLS)).astype("float32")
print("signal array:", X_sig_raw.shape)


feature columns: 14 (7 vitals + 7 masks)
signal array: (10068, 24, 14)


In [ ]:
# --- Same patient-level split + training-set standardisation as the main pipeline ---
train_ids, temp_ids = train_test_split(stay_ids, test_size=0.30, random_state=42, stratify=y.loc[stay_ids])
val_ids, test_ids   = train_test_split(temp_ids, test_size=0.50, random_state=42, stratify=y.loc[temp_ids])
pos = {s:i for i,s in enumerate(stay_ids)}; idx = lambda ids:[pos[s] for s in ids]

n_vitals = len(VITAL_COLS)
tr_vals = X_sig_raw[idx(train_ids)][:, :, :n_vitals].reshape(-1, n_vitals)
mu = np.nanmean(tr_vals, axis=0); sd = np.nanstd(tr_vals, axis=0); sd[sd==0]=1
X_sig = X_sig_raw.copy()
X_sig[:, :, :n_vitals] = (X_sig[:, :, :n_vitals] - mu) / sd
X_sig = np.nan_to_num(X_sig, nan=0.0)

def make(ids):
    i = idx(ids)
    return (torch.tensor(X_sig[i]), torch.tensor(X_text_all[i]),
            torch.tensor(has_note_all[i]), torch.tensor(y.loc[ids].values.astype("float32")))
Xs_tr,Xt_tr,m_tr,y_tr = make(train_ids)
Xs_va,Xt_va,m_va,y_va = make(val_ids)
Xs_te,Xt_te,m_te,y_te = make(test_ids)
tr_loader = DataLoader(TensorDataset(Xs_tr,Xt_tr,m_tr,y_tr), batch_size=128, shuffle=True)
print("train/val/test:", len(train_ids), len(val_ids), len(test_ids))


train/val/test: 7047 1510 1511


In [ ]:
# --- Model definitions (identical to the main notebooks; sig_dim auto = 14) ---
class LSTMClassifier(nn.Module):
    def __init__(self, n_features=len(FEAT_COLS), hidden=64, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers>1 else 0.0)
        self.dropout = nn.Dropout(dropout); self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out,(h_n,_) = self.lstm(x); return self.fc(self.dropout(h_n[-1])).squeeze(1)

class ConcatFusion(nn.Module):
    def __init__(self, sig_dim=len(FEAT_COLS), text_dim=768, d=128, dropout=0.3):
        super().__init__()
        self.sig_lstm = nn.LSTM(sig_dim, d, batch_first=True)
        self.text_proj = nn.Linear(text_dim, d)
        self.fc = nn.Sequential(nn.Linear(d*2,64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64,1))
    def forward(self, sig, text, note_mask):
        s,_ = self.sig_lstm(sig); s_rep = s.mean(1)
        t = self.text_proj(text); m = note_mask.unsqueeze(-1)
        t_rep = (t*m).sum(1)/m.sum(1).clamp(min=1)
        return self.fc(torch.cat([s_rep,t_rep],1)).squeeze(1)

class CrossAttnFusion(nn.Module):
    def __init__(self, sig_dim=len(FEAT_COLS), text_dim=768, d=128, heads=4, dropout=0.3):
        super().__init__()
        self.sig_lstm = nn.LSTM(sig_dim, d, batch_first=True)
        self.text_proj = nn.Linear(text_dim, d)
        self.cross = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(dropout)
        self.fc = nn.Sequential(nn.Linear(d,64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64,1))
    def forward(self, sig, text, note_mask):
        s,_ = self.sig_lstm(sig); t = self.text_proj(text)
        key_pad = note_mask==0; all_zero = key_pad.all(dim=1)
        key_pad = key_pad.clone(); key_pad[all_zero]=False
        attn_out,_ = self.cross(s, t, t, key_padding_mask=key_pad)
        fused = self.norm(s + self.drop(attn_out))
        return self.fc(fused.mean(1)).squeeze(1)


In [ ]:
# --- Training helpers ---
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

pr = float(y_tr.mean()); pw = torch.tensor([(1-pr)/pr], device=device)

def train_eval_signal():
    set_seed(42)
    model = LSTMClassifier().to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    tl = DataLoader(TensorDataset(Xs_tr,y_tr), batch_size=128, shuffle=True)
    best,bs=0,None
    for ep in range(1,41):
        model.train()
        for xb,yb in tl:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(); crit(model(xb),yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va=roc_auc_score(y_va.numpy(), torch.sigmoid(model(Xs_va.to(device))).cpu().numpy())
        if va>best: best,bs=va,{k:v.cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bs); model.eval()
    with torch.no_grad():
        p=torch.sigmoid(model(Xs_te.to(device))).cpu().numpy()
    yt=y_te.numpy()
    return roc_auc_score(yt,p), average_precision_score(yt,p), f1_score(yt,(p>=0.5).astype(int))

def train_eval_fusion(ModelClass, cross=False):
    set_seed(42)
    model = ModelClass().to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    best,bs=0,None
    for ep in range(1,61):
        model.train()
        for xs,xt,mm,yb in tr_loader:
            xs,xt,mm,yb=xs.to(device),xt.to(device),mm.to(device),yb.to(device)
            opt.zero_grad(); out=model(xs,xt,mm)
            crit(out,yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va=roc_auc_score(y_va.numpy(), torch.sigmoid(model(Xs_va.to(device),Xt_va.to(device),m_va.to(device))).cpu().numpy())
        if va>best: best,bs=va,{k:v.cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bs); model.eval()
    with torch.no_grad():
        p=torch.sigmoid(model(Xs_te.to(device),Xt_te.to(device),m_te.to(device))).cpu().numpy()
    yt=y_te.numpy()
    return roc_auc_score(yt,p), average_precision_score(yt,p), f1_score(yt,(p>=0.5).astype(int))


In [ ]:
# --- Run all three 7-variable models (seed 42) ---
sig = train_eval_signal();               print(f"signal-only     AUROC {sig[0]:.3f} AUPRC {sig[1]:.3f} F1 {sig[2]:.3f}")
con = train_eval_fusion(ConcatFusion);   print(f"concat fusion   AUROC {con[0]:.3f} AUPRC {con[1]:.3f} F1 {con[2]:.3f}")
cro = train_eval_fusion(CrossAttnFusion);print(f"cross-attention AUROC {cro[0]:.3f} AUPRC {cro[1]:.3f} F1 {cro[2]:.3f}")

print("\n=== Table B.1 (7-variable column) ===")
print(f"{'model':<26}{'AUROC':<8}")
print(f"{'signal-only':<26}{sig[0]:<8.3f}")
print(f"{'text-only (unchanged)':<26}{'0.669':<8}")
print(f"{'concat fusion':<26}{con[0]:<8.3f}")
print(f"{'cross-attention fusion':<26}{cro[0]:<8.3f}")


signal-only     AUROC 0.652 AUPRC 0.350 F1 0.378
concat fusion   AUROC 0.790 AUPRC 0.516 F1 0.535
cross-attention AUROC 0.794 AUPRC 0.509 F1 0.523

=== Table B.1 (7-variable column) ===
model                     AUROC   
signal-only               0.652   
text-only (unchanged)     0.669   
concat fusion             0.790   
cross-attention fusion    0.794   
